In [24]:
from config.paths import (
    PROCESSED_DIR,
    ORIG_PARQUET_DIR,
    PERF_PARQUET_DIR,
    AUDIT_DIR,
    DUCKDB_TEMP_DIR,
    get_vintage_file
)

In [8]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path(r"D:\DS Project\credit-portfolio-risk-modeling")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import duckdb

from config.paths import (
    AUDIT_DIR,
    ORIG_PARQUET_DIR,
    PERF_PARQUET_DIR,
    DUCKDB_TEMP_DIR,
    get_vintage_file
)

# Load field mappings created in Notebook 02
orig_mapping = pd.read_csv(
    AUDIT_DIR / "origination_mapping.csv"
)

perf_mapping = pd.read_csv(
    AUDIT_DIR / "performance_mapping.csv"
)

print("Origination mapping:", orig_mapping.shape)
print("Performance mapping:", perf_mapping.shape)
print("DuckDB version:", duckdb.__version__)

Origination mapping: (31, 7)
Performance mapping: (35, 7)
DuckDB version: 2.0.0


In [9]:
print("ORIGINATION MAPPING")
display(
    orig_mapping[
        ["Field Position", "standard_name"]
    ].sort_values("Field Position")
)

print("\nPERFORMANCE MAPPING")
display(
    perf_mapping[
        ["Field Position", "standard_name"]
    ].sort_values("Field Position")
)

ORIGINATION MAPPING


,Field Position,standard_name
0,1,classic_fico
1,2,first_payment_date
2,3,first_time_homebuyer_indicator
3,4,maturity_date
4,5,metropolitan_statistical_area_msa_or_metropoli...
5,6,mortgage_insurance_percentage_mi
6,7,number_of_units
7,8,occupancy_status
8,9,original_combined_loan_to_value_cltv
9,10,original_debt_to_income_dti_ratio



PERFORMANCE MAPPING


,Field Position,standard_name
0,1,loan_identifier
1,2,period
2,3,current_actual_upb
3,4,current_loan_delinquency_status
4,5,loan_age
5,6,remaining_months_to_legal_maturity
6,7,underwriting_defect_and_major_servicing_defect...
7,8,modification_flag
8,9,zero_balance_code
9,10,zero_balance_effective_date


In [11]:
# Sort mappings by original field position
orig_sorted = orig_mapping.sort_values("Field Position")
perf_sorted = perf_mapping.sort_values("Field Position")

# Create DuckDB column definitions
orig_columns = {
    name: "VARCHAR"
    for name in orig_sorted["standard_name"]
}

perf_columns = {
    name: "VARCHAR"
    for name in perf_sorted["standard_name"]
}

# Validate schema
assert len(orig_columns) == 31
assert len(perf_columns) == 35

assert len(orig_columns) == len(set(orig_columns))
assert len(perf_columns) == len(set(perf_columns))

print("Origination columns:", len(orig_columns))
print("Performance columns:", len(perf_columns))

print("\nOrigination column names:")
print(list(orig_columns.keys()))

print("\nPerformance column names:")
print(list(perf_columns.keys()))

Origination columns: 31
Performance columns: 35

Origination column names:
['classic_fico', 'first_payment_date', 'first_time_homebuyer_indicator', 'maturity_date', 'metropolitan_statistical_area_msa_or_metropolitan_division', 'mortgage_insurance_percentage_mi', 'number_of_units', 'occupancy_status', 'original_combined_loan_to_value_cltv', 'original_debt_to_income_dti_ratio', 'original_upb', 'original_loan_to_value_ltv', 'original_interest_rate', 'channel', 'prepayment_penalty_indicator', 'amortization_type', 'property_state', 'property_type', 'postal_code', 'loan_identifier', 'loan_purpose', 'original_loan_term', 'number_of_borrowers', 'seller_name', 'super_conforming_flag', 'pre_harp_loan_sequence_number', 'special_eligibility_program', 'harp_indicator', 'property_valuation_method', 'interest_only_i_o_indicator', 'vantagescore_4_0']

Performance column names:
['loan_identifier', 'period', 'current_actual_upb', 'current_loan_delinquency_status', 'loan_age', 'remaining_months_to_legal_

In [14]:
# Locate the test file
orig_file = get_vintage_file("orig", 2012, 4)

print("File:", orig_file)
print("File size (GB):", round(orig_file.stat().st_size / 1024**3, 3))

con = duckdb.connect()

try:
    orig_sample = con.read_csv(
        str(orig_file),
        delimiter="|",  # Changed from delim to delimiter
        header=False,
        columns=orig_columns,
        sample_size=20480,
        ignore_errors=False
    )

    test_df = orig_sample.limit(5).df()

    print("Number of columns:", len(test_df.columns))
    display(test_df)

finally:
    con.close()

File: E:\DS_project_backup\credit-risk-portfolio\data\orig_2012Q4.txt
File size (GB): 0.071
Number of columns: 31


,classic_fico,first_payment_date,first_time_homebuyer_indicator,maturity_date,metropolitan_statistical_area_msa_or_metropolitan_division,mortgage_insurance_percentage_mi,number_of_units,occupancy_status,original_combined_loan_to_value_cltv,original_debt_to_income_dti_ratio,...,original_loan_term,number_of_borrowers,seller_name,super_conforming_flag,pre_harp_loan_sequence_number,special_eligibility_program,harp_indicator,property_valuation_method,interest_only_i_o_indicator,vantagescore_4_0
0,804,201301,N,202712,NaN,0,1,P,71,18,...,180,2,OTHER,N,NaN,None,N,7,N,9999
1,715,201302,N,204301,37900,0,1,P,71,26,...,360,2,OTHER,N,NaN,None,N,7,N,9999
2,710,201212,N,204211,33340,0,1,P,80,45,...,360,2,OTHER,N,NaN,None,N,7,N,9999
3,776,201212,N,204211,31740,30,1,P,95,31,...,360,2,OTHER,N,NaN,None,N,7,N,9999
4,755,201212,N,204211,16974,0,1,P,106,999,...,360,1,OTHER,N,F09Q20008428,None,Y,7,N,9999


In [15]:
# Read the first raw line directly
with open(orig_file, "r", encoding="utf-8") as f:
    first_raw_line = f.readline().rstrip("\r\n")

raw_values = first_raw_line.split("|")

print("Number of raw fields:", len(raw_values))
print("Number of mapped fields:", len(orig_columns))

assert len(raw_values) == 31
assert len(orig_columns) == 31

# Compare raw values with DuckDB output
comparison = pd.DataFrame({
    "Field Position": range(1, 32),
    "Column Name": list(orig_columns.keys()),
    "Raw Value": raw_values,
    "DuckDB Value": test_df.iloc[0].astype(str).tolist()
})

display(comparison)

Number of raw fields: 31
Number of mapped fields: 31


,Field Position,Column Name,Raw Value,DuckDB Value
0,1,classic_fico,804,804
1,2,first_payment_date,201301,201301
2,3,first_time_homebuyer_indicator,N,N
3,4,maturity_date,202712,202712
4,5,metropolitan_statistical_area_msa_or_metropoli...,,NaN
5,6,mortgage_insurance_percentage_mi,0,0
6,7,number_of_units,1,1
7,8,occupancy_status,P,P
8,9,original_combined_loan_to_value_cltv,71,71
9,10,original_debt_to_income_dti_ratio,18,18


In [16]:
from pathlib import Path

# Define input and output
orig_file = get_vintage_file("orig", 2012, 4)

output_file = ORIG_PARQUET_DIR / "orig_2012Q4.parquet"

# Prevent accidental overwrite
if output_file.exists():
    raise FileExistsError(
        f"Output already exists: {output_file}"
    )

# Connect to DuckDB
con = duckdb.connect()

try:
    # Configure temporary storage
    temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")

    con.execute(
        f"SET temp_directory = '{temp_path}'"
    )

    # Read TXT using the audited schema
    orig_relation = con.read_csv(
        str(orig_file),
        delimiter="|",
        header=False,
        columns=orig_columns,
        ignore_errors=False
    )

    # Write directly to Parquet
    orig_relation.write_parquet(
        str(output_file),
        compression="zstd"
    )

    # Validate output
    row_count = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [str(output_file)]
    ).fetchone()[0]

    print("Ingestion completed successfully.")
    print("Output:", output_file)
    print("Rows:", f"{row_count:,}")
    print(
        "Parquet size (MB):",
        round(output_file.stat().st_size / 1024**2, 2)
    )

finally:
    con.close()

Ingestion completed successfully.
Output: E:\DS_project_backup\credit-risk-portfolio\processed\origination\orig_2012Q4.parquet
Rows: 592,713
Parquet size (MB): 7.14


In [17]:
con = duckdb.connect()

try:
    parquet_check = con.execute(
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT loan_identifier) AS unique_loans,
            MIN(original_upb) AS min_original_upb,
            MAX(original_upb) AS max_original_upb,
            COUNT(*) FILTER (
                WHERE original_upb IS NULL
            ) AS missing_original_upb
        FROM read_parquet(?)
        """,
        [str(output_file)]
    ).df()

    display(parquet_check)

finally:
    con.close()

,total_rows,unique_loans,min_original_upb,max_original_upb,missing_original_upb
0,592713,592713,100000,99000,0


In [18]:
con = duckdb.connect()

try:
    numeric_check = con.execute(
        """
        SELECT
            COUNT(*) AS total_rows,

            COUNT(DISTINCT loan_identifier) AS unique_loans,

            COUNT(*) FILTER (
                WHERE loan_identifier IS NULL
            ) AS missing_loan_identifier,

            MIN(TRY_CAST(original_upb AS DOUBLE))
                AS min_original_upb,

            MAX(TRY_CAST(original_upb AS DOUBLE))
                AS max_original_upb,

            AVG(TRY_CAST(original_upb AS DOUBLE))
                AS avg_original_upb,

            COUNT(*) FILTER (
                WHERE TRY_CAST(original_upb AS DOUBLE) IS NULL
            ) AS invalid_original_upb

        FROM read_parquet(?)
        """,
        [str(output_file)]
    ).df()

    print(numeric_check.to_string(index=False))

finally:
    con.close()

 total_rows  unique_loans  missing_loan_identifier  min_original_upb  max_original_upb  avg_original_upb  invalid_original_upb
     592713        592713                        0            7000.0         1200000.0     205580.220107                     0


In [19]:
con = duckdb.connect()

try:
    parquet_schema = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [str(output_file)]
    ).df()

    display(parquet_schema)

finally:
    con.close()

,column_name,column_type,null,key,default,extra
0,classic_fico,VARCHAR,YES,None,None,None
1,first_payment_date,VARCHAR,YES,None,None,None
2,first_time_homebuyer_indicator,VARCHAR,YES,None,None,None
3,maturity_date,VARCHAR,YES,None,None,None
4,metropolitan_statistical_area_msa_or_metropoli...,VARCHAR,YES,None,None,None
5,mortgage_insurance_percentage_mi,VARCHAR,YES,None,None,None
6,number_of_units,VARCHAR,YES,None,None,None
7,occupancy_status,VARCHAR,YES,None,None,None
8,original_combined_loan_to_value_cltv,VARCHAR,YES,None,None,None
9,original_debt_to_income_dti_ratio,VARCHAR,YES,None,None,None


In [21]:
from pathlib import Path

for vintage in ["2006Q1", "2006Q2", "2012Q4"]:
    path = ORIG_PARQUET_DIR / f"orig_{vintage}.parquet"

    print(
        vintage,
        "| Exists:", path.exists(),
        "| Size (MB):",
        round(path.stat().st_size / 1024**2, 2)
        if path.exists() else "N/A"
    )

2006Q1 | Exists: True | Size (MB): 3.12
2006Q2 | Exists: False | Size (MB): N/A
2012Q4 | Exists: True | Size (MB): 7.14


In [22]:
import gc
import pandas as pd
import duckdb

ingestion_log = []

for year in range(2006, 2013):
    for quarter in range(1, 5):

        vintage = f"{year}Q{quarter}"

        input_file = get_vintage_file(
            "orig", year, quarter
        )

        output_file = (
            ORIG_PARQUET_DIR / f"orig_{vintage}.parquet"
        )

        temp_output = output_file.with_name(
            output_file.stem + ".tmp.parquet"
        )

        print(f"\nProcessing {vintage}...")

        # Skip existing files, including previously validated files
        if output_file.exists():

            con = duckdb.connect()

            try:
                row_count = con.execute(
                    "SELECT COUNT(*) FROM read_parquet(?)",
                    [str(output_file)]
                ).fetchone()[0]
            finally:
                con.close()

            status = "skipped_existing"

        else:

            if temp_output.exists():
                temp_output.unlink()

            con = duckdb.connect()
            relation = None

            try:
                temp_path = (
                    DUCKDB_TEMP_DIR.as_posix()
                    .replace("'", "''")
                )

                con.execute(
                    f"SET temp_directory = '{temp_path}'"
                )

                relation = con.read_csv(
                    str(input_file),
                    delimiter="|",
                    header=False,
                    columns=orig_columns,
                    ignore_errors=False
                )

                relation.write_parquet(
                    str(temp_output),
                    compression="zstd"
                )

                # Release the relation before closing connection
                del relation
                relation = None
                gc.collect()

                # Validate output
                row_count = con.execute(
                    "SELECT COUNT(*) FROM read_parquet(?)",
                    [str(temp_output)]
                ).fetchone()[0]

                # Release query result
                con.execute("CHECKPOINT")

                # Publish completed file
                temp_output.replace(output_file)

                status = "completed"

            except Exception:
                if temp_output.exists():
                    temp_output.unlink()
                raise

            finally:
                if relation is not None:
                    del relation

                gc.collect()
                con.close()

        # Record results
        record = {
            "vintage": vintage,
            "rows": row_count,
            "input_size_mb": round(
                input_file.stat().st_size / 1024**2, 2
            ),
            "output_size_mb": round(
                output_file.stat().st_size / 1024**2, 2
            ),
            "status": status
        }

        ingestion_log.append(record)

        print(f"Status: {status}")
        print(f"Rows: {row_count:,}")

        # Save manifest after each vintage
        ingestion_manifest = pd.DataFrame(ingestion_log)

        ingestion_manifest.to_csv(
            AUDIT_DIR / "origination_ingestion_manifest.csv",
            index=False,
            encoding="utf-8-sig"
        )

print("\nAll Origination vintages processed.")
display(ingestion_manifest)

print(
    "Total loans:",
    f"{ingestion_manifest['rows'].sum():,}"
)


Processing 2006Q1...
Status: skipped_existing
Rows: 298,574

Processing 2006Q2...
Status: completed
Rows: 308,208

Processing 2006Q3...
Status: completed
Rows: 283,975

Processing 2006Q4...
Status: completed
Rows: 302,787

Processing 2007Q1...
Status: completed
Rows: 302,331

Processing 2007Q2...
Status: completed
Rows: 354,055

Processing 2007Q3...
Status: completed
Rows: 284,980

Processing 2007Q4...
Status: completed
Rows: 296,826

Processing 2008Q1...
Status: completed
Rows: 421,635

Processing 2008Q2...
Status: completed
Rows: 370,982

Processing 2008Q3...
Status: completed
Rows: 221,752

Processing 2008Q4...
Status: completed
Rows: 218,777

Processing 2009Q1...
Status: completed
Rows: 621,542

Processing 2009Q2...
Status: completed
Rows: 716,665

Processing 2009Q3...
Status: completed
Rows: 443,064

Processing 2009Q4...
Status: completed
Rows: 445,753

Processing 2010Q1...
Status: completed
Rows: 360,856

Processing 2010Q2...
Status: completed
Rows: 363,854

Processing 2010Q3...

,vintage,rows,input_size_mb,output_size_mb,status
0,2006Q1,298574,36.46,3.12,skipped_existing
1,2006Q2,308208,37.62,3.27,completed
2,2006Q3,283975,34.75,3.07,completed
3,2006Q4,302787,37.12,3.28,completed
4,2007Q1,302331,37.05,3.24,completed
5,2007Q2,354055,43.35,3.80,completed
6,2007Q3,284980,34.61,3.11,completed
7,2007Q4,296826,35.95,3.23,completed
8,2008Q1,421635,51.41,4.47,completed
9,2008Q2,370982,45.12,3.85,completed


Total loans: 11,128,442


In [25]:
import shutil

# Locate Performance 2012Q4
perf_file = get_vintage_file("perf", 2012, 4)

# Check file size
perf_size_gb = perf_file.stat().st_size / 1024**3

# Check free disk space
disk_usage = shutil.disk_usage(str(perf_file.parent))
free_space_gb = disk_usage.free / 1024**3

# Check processed storage
processed_usage = shutil.disk_usage(str(PROCESSED_DIR))
processed_free_gb = processed_usage.free / 1024**3

print("Performance file:", perf_file)
print("Raw file size (GB):", round(perf_size_gb, 2))

print("\nE drive free space (GB):", round(free_space_gb, 2))
print("Processed storage free space (GB):", round(processed_free_gb, 2))

print("\nExpected columns:", len(perf_columns))

Performance file: E:\DS_project_backup\credit-risk-portfolio\data\perf_2012Q4.txt
Raw file size (GB): 5.95

E drive free space (GB): 52.56
Processed storage free space (GB): 52.56

Expected columns: 35


In [26]:
# Read only the first raw line
with open(perf_file, "r", encoding="utf-8") as f:
    first_perf_line = f.readline().rstrip("\r\n")

perf_raw_values = first_perf_line.split("|")

print("Number of raw fields:", len(perf_raw_values))
print("Number of mapped fields:", len(perf_columns))

assert len(perf_raw_values) == 35
assert len(perf_columns) == 35

print("\nFirst 10 raw values:")
print(perf_raw_values[:10])

Number of raw fields: 35
Number of mapped fields: 35

First 10 raw values:
['F12Q40000001', '201212', '156000.00', '00', '0', '180', '', '', '', '']


In [27]:
con = duckdb.connect()
perf_relation = None

try:
    perf_relation = con.read_csv(
        str(perf_file),
        delimiter="|",
        header=False,
        columns=perf_columns,
        ignore_errors=False
    )

    perf_test_df = perf_relation.limit(5).df()

    print("Number of columns:", len(perf_test_df.columns))
    display(perf_test_df)

finally:
    if perf_relation is not None:
        del perf_relation

    con.close()

Number of columns: 35


,loan_identifier,period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity,underwriting_defect_and_major_servicing_defect_settlement_date,modification_flag,zero_balance_code,zero_balance_effective_date,...,estimated_loan_to_value_eltv,zero_balance_removal_upb,delinquent_accrued_interest,delinquency_due_to_disaster,borrower_assistance_plan,current_period_modification_costs,current_interest_bearing_upb,mortgage_insurance_cancellation_indicator,servicer_name,bankruptcy_cramdown_costs
0,F12Q40000001,201212,156000.00,00,0,180,None,None,None,None,...,999,None,None,None,None,None,156000.00,7,OTHER,None
1,F12Q40000001,201301,155000.00,00,1,179,None,None,None,None,...,999,None,None,None,None,None,155000.00,7,OTHER,None
2,F12Q40000001,201302,155000.00,00,2,178,None,None,None,None,...,999,None,None,None,None,None,155000.00,7,OTHER,None
3,F12Q40000001,201303,154000.00,00,3,177,None,None,None,None,...,999,None,None,None,None,None,154000.00,7,OTHER,None
4,F12Q40000001,201304,153000.00,00,4,176,None,None,None,None,...,999,None,None,None,None,None,153000.00,7,OTHER,None


In [28]:
perf_comparison_test = pd.DataFrame({
    "Field Position": range(1, 36),
    "Column Name": list(perf_columns.keys()),
    "Raw Value": perf_raw_values,
    "DuckDB Value": perf_test_df.iloc[0].astype(str).tolist()
})

display(perf_comparison_test)

,Field Position,Column Name,Raw Value,DuckDB Value
0,1,loan_identifier,F12Q40000001,F12Q40000001
1,2,period,201212,201212
2,3,current_actual_upb,156000.00,156000.00
3,4,current_loan_delinquency_status,00,00
4,5,loan_age,0,0
5,6,remaining_months_to_legal_maturity,180,180
6,7,underwriting_defect_and_major_servicing_defect...,,NaN
7,8,modification_flag,,NaN
8,9,zero_balance_code,,NaN
9,10,zero_balance_effective_date,,NaN


In [29]:
import gc
import duckdb

# Input and output
perf_file = get_vintage_file("perf", 2012, 4)

output_file = (
    PERF_PARQUET_DIR / "perf_2012Q4.parquet"
)

temp_output = (
    PERF_PARQUET_DIR / "perf_2012Q4.tmp.parquet"
)

# Prevent accidental overwrite
if output_file.exists():
    raise FileExistsError(
        f"Output already exists: {output_file}"
    )

if temp_output.exists():
    temp_output.unlink()

con = duckdb.connect()
perf_relation = None

try:
    # Configure temporary storage
    temp_path = (
        DUCKDB_TEMP_DIR.as_posix()
        .replace("'", "''")
    )

    con.execute(
        f"SET temp_directory = '{temp_path}'"
    )

    print("Reading Performance TXT...")

    perf_relation = con.read_csv(
        str(perf_file),
        delimiter="|",
        header=False,
        columns=perf_columns,
        ignore_errors=False
    )

    print("Writing Parquet...")

    perf_relation.write_parquet(
        str(temp_output),
        compression="zstd"
    )

    # Release relation before closing connection
    del perf_relation
    perf_relation = None
    gc.collect()

    print("Validating Parquet...")

    row_count = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [str(temp_output)]
    ).fetchone()[0]

    # Publish only after successful validation
    temp_output.replace(output_file)

    print("\nIngestion completed successfully.")
    print("Rows:", f"{row_count:,}")

    print(
        "Raw size (GB):",
        round(perf_file.stat().st_size / 1024**3, 2)
    )

    print(
        "Parquet size (GB):",
        round(output_file.stat().st_size / 1024**3, 3)
    )

finally:
    if perf_relation is not None:
        del perf_relation

    gc.collect()
    con.close()

Reading Performance TXT...
Writing Parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Validating Parquet...

Ingestion completed successfully.
Rows: 56,807,882
Raw size (GB): 5.95
Parquet size (GB): 0.527


In [30]:
con = duckdb.connect()

try:
    perf_check = con.execute(
        """
        SELECT
            COUNT(*) AS total_records,

            COUNT(DISTINCT loan_identifier)
                AS unique_loans,

            COUNT(*) FILTER (
                WHERE loan_identifier IS NULL
            ) AS missing_loan_identifier,

            MIN(period) AS first_reporting_period,

            MAX(period) AS last_reporting_period,

            COUNT(*) FILTER (
                WHERE current_actual_upb IS NULL
            ) AS missing_current_upb

        FROM read_parquet(?)
        """,
        [str(output_file)]
    ).df()

    display(perf_check)

finally:
    con.close()

,total_records,unique_loans,missing_loan_identifier,first_reporting_period,last_reporting_period,missing_current_upb
0,56807882,592713,0,201210,202606,0


In [31]:
import shutil

disk = shutil.disk_usage(
    r"E:\DS_project_backup\credit-risk-portfolio"
)

print("Free space (GB):", round(disk.free / 1024**3, 2))

print(
    "Existing Performance Parquet (GB):",
    round(
        sum(
            p.stat().st_size
            for p in PERF_PARQUET_DIR.glob("*.parquet")
        ) / 1024**3,
        3
    )
)

Free space (GB): 52.03
Existing Performance Parquet (GB): 0.527


In [32]:
import gc
import pandas as pd
import duckdb
import shutil

# Initialize ingestion log
perf_ingestion_log = []

manifest_path = (
    AUDIT_DIR / "performance_ingestion_manifest.csv"
)

# Process all 28 quarterly vintages
for year in range(2006, 2013):
    for quarter in range(1, 5):

        vintage = f"{year}Q{quarter}"

        input_file = get_vintage_file(
            "perf", year, quarter
        )

        output_file = (
            PERF_PARQUET_DIR /
            f"perf_{vintage}.parquet"
        )

        temp_output = output_file.with_name(
            output_file.stem + ".tmp.parquet"
        )

        print(f"\nProcessing {vintage}...")

        # Check available disk space
        disk = shutil.disk_usage(
            r"E:\DS_project_backup\credit-risk-portfolio"
        )

        free_gb = disk.free / 1024**3

        if free_gb < 10:
            raise RuntimeError(
                f"Stopping ingestion: only "
                f"{free_gb:.2f} GB free."
            )

        # Skip existing output
        if output_file.exists():

            con = duckdb.connect()

            try:
                row_count = con.execute(
                    "SELECT COUNT(*) "
                    "FROM read_parquet(?)",
                    [str(output_file)]
                ).fetchone()[0]

            finally:
                con.close()

            status = "skipped_existing"

        else:

            if temp_output.exists():
                temp_output.unlink()

            con = duckdb.connect()
            relation = None

            try:
                temp_path = (
                    DUCKDB_TEMP_DIR.as_posix()
                    .replace("'", "''")
                )

                con.execute(
                    f"SET temp_directory = '{temp_path}'"
                )

                relation = con.read_csv(
                    str(input_file),
                    delimiter="|",
                    header=False,
                    columns=perf_columns,
                    ignore_errors=False
                )

                relation.write_parquet(
                    str(temp_output),
                    compression="zstd"
                )

                # Release relation
                del relation
                relation = None
                gc.collect()

                # Validate Parquet row count
                row_count = con.execute(
                    "SELECT COUNT(*) "
                    "FROM read_parquet(?)",
                    [str(temp_output)]
                ).fetchone()[0]

                # Publish validated output
                temp_output.replace(output_file)

                status = "completed"

            except Exception:
                if temp_output.exists():
                    temp_output.unlink()
                raise

            finally:
                if relation is not None:
                    del relation

                gc.collect()
                con.close()

        # Record results
        record = {
            "vintage": vintage,
            "rows": row_count,
            "input_size_gb": round(
                input_file.stat().st_size / 1024**3, 3
            ),
            "output_size_gb": round(
                output_file.stat().st_size / 1024**3, 3
            ),
            "status": status
        }

        perf_ingestion_log.append(record)

        # Save manifest after every vintage
        pd.DataFrame(
            perf_ingestion_log
        ).to_csv(
            manifest_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Status: {status}")
        print(f"Records: {row_count:,}")
        print(f"Free disk space: {free_gb:.2f} GB")

print("\nPerformance batch ingestion finished.")

perf_ingestion_manifest = pd.DataFrame(
    perf_ingestion_log
)

display(perf_ingestion_manifest)

print(
    "Total Performance records:",
    f"{perf_ingestion_manifest['rows'].sum():,}"
)

print("Manifest:", manifest_path)


Processing 2006Q1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Status: completed
Records: 20,641,773
Free disk space: 52.03 GB

Processing 2006Q2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Status: completed
Records: 19,232,256
Free disk space: 51.85 GB

Processing 2006Q3...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (47 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Status: completed
Records: 52,731,575
Free disk space: 46.03 GB

Processing 2012Q4...
Status: skipped_existing
Records: 56,807,882
Free disk space: 45.54 GB

Performance batch ingestion finished.


,vintage,rows,input_size_gb,output_size_gb,status
0,2006Q1,20641773,2.227,0.188,completed
1,2006Q2,19232256,2.069,0.176,completed
2,2006Q3,17091022,1.837,0.157,completed
3,2006Q4,19600060,2.116,0.182,completed
4,2007Q1,19455523,2.100,0.181,completed
5,2007Q2,22314866,2.405,0.207,completed
6,2007Q3,15847264,1.702,0.148,completed
7,2007Q4,16954114,1.819,0.159,completed
8,2008Q1,23849747,2.555,0.220,completed
9,2008Q2,19540165,2.084,0.178,completed


Total Performance records: 769,725,294
Manifest: E:\DS_project_backup\credit-risk-portfolio\processed\audit\performance_ingestion_manifest.csv


In [33]:
import pandas as pd
import shutil

# Load ingestion manifests
orig_manifest = pd.read_csv(
    AUDIT_DIR / "origination_ingestion_manifest.csv"
)

perf_manifest = pd.read_csv(
    AUDIT_DIR / "performance_ingestion_manifest.csv"
)

# Expected vintages
expected_vintages = [
    f"{year}Q{quarter}"
    for year in range(2006, 2013)
    for quarter in range(1, 5)
]

# Check file counts
orig_files = list(ORIG_PARQUET_DIR.glob("orig_*.parquet"))
perf_files = list(PERF_PARQUET_DIR.glob("perf_*.parquet"))

print("ORIGINATION")
print("Manifest rows:", len(orig_manifest))
print("Parquet files:", len(orig_files))
print("Total loans:", f"{orig_manifest['rows'].sum():,}")

print("\nPERFORMANCE")
print("Manifest rows:", len(perf_manifest))
print("Parquet files:", len(perf_files))
print("Total records:", f"{perf_manifest['rows'].sum():,}")

# Check completeness
assert len(orig_manifest) == 28
assert len(perf_manifest) == 28

assert len(orig_files) == 28
assert len(perf_files) == 28

assert set(orig_manifest["vintage"]) == set(expected_vintages)
assert set(perf_manifest["vintage"]) == set(expected_vintages)

assert not orig_manifest["status"].str.contains(
    "failed", case=False
).any()

assert not perf_manifest["status"].str.contains(
    "failed", case=False
).any()

# Disk space
disk = shutil.disk_usage(
    r"E:\DS_project_backup\credit-risk-portfolio"
)

print("\nFree disk space (GB):", round(disk.free / 1024**3, 2))

print("\nAll ingestion completeness checks passed.")

ORIGINATION
Manifest rows: 28
Parquet files: 28
Total loans: 11,128,442

PERFORMANCE
Manifest rows: 28
Parquet files: 28
Total records: 769,725,294

Free disk space (GB): 45.54

All ingestion completeness checks passed.
